# Module A Deep Walkthrough: B+ Tree + WAL + Recovery

> Scenario is aligned with Section 5.1 in ASSIGNMENT_3_MODULE_A_PLAN.md (accept-offer flow with multi-table side effects).

> Goal: make the internal mechanism visible with strong before/after prints and WAL inspection.

This notebook demonstrates:

1. B+ Tree structure behavior (splits, separator keys, leaf chaining).
2. A realistic transactional use case across multiple tables:
   - accept selected offer
   - auto-decline competing offers
   - mark listing sold
   - clear watchlist rows for the sold listing
   - create transaction audit rows
   - create notification rows
3. WAL evidence (compact and raw JSONL)
4. Recovery replay (REDO committed and UNDO crash-uncommitted) on a fresh database instance.

## How To Read This Notebook

1. Run cells top to bottom once.
2. Observe printed table snapshots before and after each transactional stage.
3. Compare in-memory pre-crash state with post-recovery state.
4. Use the WAL compact table and raw log lines to verify operation ordering.

Recovery expectation in this demo:

- Fully committed transactions remain (REDO).
- Uncommitted crash-window transaction is rolled back logically (UNDO).
- Explicitly rolled-back transactions would not be treated as crash-uncommitted (not demonstrated in this specific run).

In [25]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

# Robust path setup so imports work even if notebook is opened from another CWD.
cwd = Path.cwd()
module_a_root = cwd
if not (module_a_root / "database").exists():
    for p in [cwd, *cwd.parents]:
        if (p / "database").exists():
            module_a_root = p
            break

if str(module_a_root) not in sys.path:
    sys.path.insert(0, str(module_a_root))

from database import BPlusTree, DatabaseManager, RecoveryManager

demo_dir = module_a_root / "artifacts" / "notebook_demo"
demo_dir.mkdir(parents=True, exist_ok=True)
wal_path = demo_dir / "wal_module_a_51_demo.log"

# Start from a clean WAL file for deterministic output.
if wal_path.exists():
    wal_path.unlink()

def iso_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def rows_for(mgr: DatabaseManager, table_name: str):
    table, msg = mgr.get_table(DB_NAME, table_name)
    if table is None:
        raise RuntimeError(msg)
    return [dict(row) for _, row in table.get_all()]

def print_table(mgr: DatabaseManager, table_name: str, title: str | None = None, predicate=None):
    data = rows_for(mgr, table_name)
    if predicate is not None:
        data = [r for r in data if predicate(r)]
    if title:
        print(f"\n{title}")
    print(f"{table_name}: {len(data)} row(s)")
    for row in data:
        print("  ", row)

def print_listing_bundle(mgr: DatabaseManager, listing_id: int, header: str):
    print(f"\n{'=' * 80}\n{header}\n{'=' * 80}")
    print_table(mgr, "Listing", title=f"Listing snapshot for ListingID={listing_id}", predicate=lambda r: r["ListingID"] == listing_id)
    print_table(mgr, "Offer", title=f"Offer snapshot for ListingID={listing_id}", predicate=lambda r: r["ListingID"] == listing_id)
    print_table(mgr, "Watchlist", title=f"Watchlist snapshot for ListingID={listing_id}", predicate=lambda r: r["ListingID"] == listing_id)
    print_table(mgr, "Transaction", title=f"Transaction snapshot for ListingID={listing_id}", predicate=lambda r: r["ListingID"] == listing_id)
    print_table(mgr, "Notification", title=f"Notification snapshot for ListingID={listing_id}", predicate=lambda r: r["RelatedListingID"] == listing_id)

def next_int_key(mgr: DatabaseManager, table_name: str) -> int:
    table, msg = mgr.get_table(DB_NAME, table_name)
    if table is None:
        raise RuntimeError(msg)
    keys = [k for k, _ in table.get_all() if isinstance(k, int)]
    return (max(keys) if keys else 0) + 1

print(f"Module A root: {module_a_root}")
print(f"WAL path: {wal_path}")

Module A root: d:\Courses\Databases\Project\Assignment_3\Module_A
WAL path: d:\Courses\Databases\Project\Assignment_3\Module_A\artifacts\notebook_demo\wal_module_a_51_demo.log


### What This Setup Cell Did

- Configured robust imports so notebook execution works from different working directories.
- Initialized a deterministic WAL path under artifacts/notebook_demo.
- Added reusable helpers for timestamping, table snapshots, and consistent debug printing.

If this cell runs successfully, all later cells should use the same WAL file and helper functions.

## Part 1: B+ Tree Mechanics (Larger Key Set)

This warm-up uses a larger insert sequence to force more splits so the leaf-chain behavior is clear.

What to observe from output:

- Root separator keys (internal node routing).
- Left-to-right leaf chain (for range scans).
- Correct point lookup and range query results.

In [26]:
tree = BPlusTree(order=4)
insert_keys = [15, 8, 22, 5, 11, 18, 27, 3, 7, 9, 13, 16, 20, 25, 30]
for key in insert_keys:
    tree.insert(key, {"ListingID": key, "payload": f"row-{key}"})

print("Inserted keys:", insert_keys)
print("Root node type:", type(tree.root).__name__)
print("Root separator keys:", tree.root.keys)

node = tree.root
while not node.is_leaf():
    node = node.children[0]

leaf_no = 1
leaf_chain_keys = []
while node is not None:
    print(f"Leaf {leaf_no} keys -> {node.keys}")
    leaf_chain_keys.extend(node.keys)
    node = node.next
    leaf_no += 1

print("\nFlattened leaf-chain keys:", leaf_chain_keys)
print("Point search key=20:", tree.search(20))
print("Range query [9, 22]:")
pprint(tree.range_query(9, 22))

Inserted keys: [15, 8, 22, 5, 11, 18, 27, 3, 7, 9, 13, 16, 20, 25, 30]
Root node type: InternalNode
Root separator keys: [15]
Leaf 1 keys -> [3]
Leaf 2 keys -> [5, 7]
Leaf 3 keys -> [8]
Leaf 4 keys -> [9]
Leaf 5 keys -> [11, 13]
Leaf 6 keys -> [15, 16]
Leaf 7 keys -> [18, 20]
Leaf 8 keys -> [22]
Leaf 9 keys -> [25]
Leaf 10 keys -> [27, 30]

Flattened leaf-chain keys: [3, 5, 7, 8, 9, 11, 13, 15, 16, 18, 20, 22, 25, 27, 30]
Point search key=20: {'ListingID': 20, 'payload': 'row-20'}
Range query [9, 22]:
[(9, {'ListingID': 9, 'payload': 'row-9'}),
 (11, {'ListingID': 11, 'payload': 'row-11'}),
 (13, {'ListingID': 13, 'payload': 'row-13'}),
 (15, {'ListingID': 15, 'payload': 'row-15'}),
 (16, {'ListingID': 16, 'payload': 'row-16'}),
 (18, {'ListingID': 18, 'payload': 'row-18'}),
 (20, {'ListingID': 20, 'payload': 'row-20'}),
 (22, {'ListingID': 22, 'payload': 'row-22'})]


### Interpreting Part 1 Output

- The root separator keys show how internal nodes route searches.
- The leaf chain print confirms linked leaf traversal, which is why range queries are efficient in B+ trees.
- Point search validates exact-key lookup.
- Range query validates ordered scan behavior from linked leaves.

This gives confidence that table-level index operations used later are backed by correct B+ tree mechanics.

## Part 2: Section 5.1 Scenario Setup (Multi-Table, Realistic)

This section uses a larger, Module B-aligned storyline for one listing lifecycle:

1. Accept one submitted offer.
2. Auto-decline competing submitted offers on the same listing.
3. Mark listing as Sold.
4. Clear watchlist rows for that listing.
5. Insert transaction/audit rows.
6. Insert notification rows for winner, losers, and seller.

We will run this twice:

- once as a committed transaction (should survive recovery),
- once as an uncommitted crash-window transaction (should be undone by recovery).

In [27]:
DB_NAME = "CampusTrading"

schemas = {
    "Member": {
        "MemberID": int,
        "Name": str,
        "Role": str,
    },
    "Listing": {
        "ListingID": int,
        "SellerID": int,
        "Title": str,
        "Status": str,
        "LastModifiedDate": str,
    },
    "Offer": {
        "OfferID": int,
        "ListingID": int,
        "BuyerID": int,
        "OfferedPrice": float,
        "OfferStatus": str,
        "AgreedPrice": object,
        "Reason": str,
        "ResponseDate": str,
    },
    "Watchlist": {
        "WatchlistID": int,
        "ListingID": int,
        "BuyerID": int,
    },
    "Transaction": {
        "TransactionID": int,
        "ListingID": int,
        "SellerID": int,
        "BuyerID": int,
        "OfferID": int,
        "AgreedPrice": float,
        "Status": str,
        "CreatedDate": str,
    },
    "Notification": {
        "NotificationID": int,
        "RecipientID": int,
        "NotificationType": str,
        "Title": str,
        "Message": str,
        "RelatedListingID": int,
        "RelatedOfferID": int,
        "RelatedTransactionID": int,
        "CreatedDate": str,
    },
}

db = DatabaseManager(wal_path=str(wal_path))
print(db.create_database(DB_NAME))
for table_name, table_schema in schemas.items():
    key_name = next(iter(table_schema.keys()))
    print(db.create_table(DB_NAME, table_name, table_schema, order=4, search_key=key_name))

def assert_ok(result, context: str):
    ok, msg = result
    print(f"{context}: {msg}")
    if not ok:
        raise RuntimeError(msg)

def run_accept_offer_flow(
    mgr: DatabaseManager,
    tx_id: str,
    listing_id: int,
    accepted_offer_id: int,
    acting_seller_id: int,
    include_notifications: bool = True,
    include_declined_transactions: bool = True,
    clear_watchlist: bool = True,
) -> dict:
    print(f"\n--- Running accept-offer flow | tx={tx_id} | listing={listing_id} | accepted_offer={accepted_offer_id} ---")

    accepted_offer, msg = mgr.tx_get(tx_id, DB_NAME, "Offer", accepted_offer_id)
    if accepted_offer is None:
        raise RuntimeError(msg)

    if accepted_offer["ListingID"] != listing_id:
        raise RuntimeError("Accepted offer does not belong to requested listing")
    if accepted_offer["OfferStatus"] != "Submitted":
        raise RuntimeError("Accepted offer must be in Submitted status")

    listing_row, msg = mgr.tx_get(tx_id, DB_NAME, "Listing", listing_id)
    if listing_row is None:
        raise RuntimeError(msg)
    if listing_row["SellerID"] != acting_seller_id:
        raise RuntimeError("Acting seller does not own this listing")
    if listing_row["Status"] not in {"Listed", "Pending"}:
        raise RuntimeError("Listing status does not allow acceptance")

    agreed_price = float(accepted_offer["OfferedPrice"])
    accepted_updated = dict(accepted_offer)
    accepted_updated["OfferStatus"] = "Accepted"
    accepted_updated["AgreedPrice"] = agreed_price
    accepted_updated["Reason"] = "Accepted by seller"
    accepted_updated["ResponseDate"] = iso_now()
    assert_ok(
        mgr.tx_update(tx_id, DB_NAME, "Offer", accepted_offer_id, accepted_updated),
        "Step 1 (accept chosen offer)",
    )

    offer_table, _ = mgr.get_table(DB_NAME, "Offer")
    competing_offer_ids = []
    for offer_key, row in offer_table.get_all():
        if (
            row["ListingID"] == listing_id
            and offer_key != accepted_offer_id
            and row["OfferStatus"] == "Submitted"
        ):
            competing_offer_ids.append(int(offer_key))

    for other_offer_id in competing_offer_ids:
        other_offer, msg = mgr.tx_get(tx_id, DB_NAME, "Offer", other_offer_id)
        if other_offer is None:
            raise RuntimeError(msg)
        declined = dict(other_offer)
        declined["OfferStatus"] = "Declined"
        declined["Reason"] = "Sold to another buyer"
        declined["ResponseDate"] = iso_now()
        assert_ok(
            mgr.tx_update(tx_id, DB_NAME, "Offer", other_offer_id, declined),
            f"Step 2 (decline competing offer {other_offer_id})",
        )

    updated_listing = dict(listing_row)
    updated_listing["Status"] = "Sold"
    updated_listing["LastModifiedDate"] = iso_now()
    assert_ok(
        mgr.tx_update(tx_id, DB_NAME, "Listing", listing_id, updated_listing),
        "Step 3 (mark listing sold)",
    )

    removed_watchlist_ids = []
    if clear_watchlist:
        watchlist_table, _ = mgr.get_table(DB_NAME, "Watchlist")
        for watchlist_id, row in watchlist_table.get_all():
            if row["ListingID"] == listing_id:
                assert_ok(
                    mgr.tx_delete(tx_id, DB_NAME, "Watchlist", watchlist_id),
                    f"Step 4 (remove watchlist {watchlist_id})",
                )
                removed_watchlist_ids.append(int(watchlist_id))

    accepted_txn_id = next_int_key(mgr, "Transaction")
    accepted_txn = {
        "TransactionID": accepted_txn_id,
        "ListingID": listing_id,
        "SellerID": acting_seller_id,
        "BuyerID": accepted_offer["BuyerID"],
        "OfferID": accepted_offer_id,
        "AgreedPrice": agreed_price,
        "Status": "Scheduled",
        "CreatedDate": iso_now(),
    }
    assert_ok(
        mgr.tx_insert(tx_id, DB_NAME, "Transaction", accepted_txn),
        "Step 5 (insert accepted transaction row)",
    )

    declined_txn_pairs = []
    if include_declined_transactions:
        for other_offer_id in competing_offer_ids:
            other_offer, msg = mgr.tx_get(tx_id, DB_NAME, "Offer", other_offer_id)
            if other_offer is None:
                raise RuntimeError(msg)
            decline_txn_id = next_int_key(mgr, "Transaction")
            decline_txn = {
                "TransactionID": decline_txn_id,
                "ListingID": listing_id,
                "SellerID": acting_seller_id,
                "BuyerID": other_offer["BuyerID"],
                "OfferID": other_offer_id,
                "AgreedPrice": float(other_offer["OfferedPrice"]),
                "Status": "ClosedLost",
                "CreatedDate": iso_now(),
            }
            assert_ok(
                mgr.tx_insert(tx_id, DB_NAME, "Transaction", decline_txn),
                f"Step 5b (insert declined transaction for offer {other_offer_id})",
            )
            declined_txn_pairs.append((other_offer_id, decline_txn_id))

    created_notification_ids = []
    if include_notifications:
        winner_note_id = next_int_key(mgr, "Notification")
        winner_note = {
            "NotificationID": winner_note_id,
            "RecipientID": accepted_offer["BuyerID"],
            "NotificationType": "OfferAccepted",
            "Title": "Offer Accepted",
            "Message": f"Your offer {accepted_offer_id} was accepted for listing {listing_id}.",
            "RelatedListingID": listing_id,
            "RelatedOfferID": accepted_offer_id,
            "RelatedTransactionID": accepted_txn_id,
            "CreatedDate": iso_now(),
        }
        assert_ok(
            mgr.tx_insert(tx_id, DB_NAME, "Notification", winner_note),
            "Step 6 (notify winner)",
        )
        created_notification_ids.append(winner_note_id)

        seller_note_id = next_int_key(mgr, "Notification")
        seller_note = {
            "NotificationID": seller_note_id,
            "RecipientID": acting_seller_id,
            "NotificationType": "TransactionCompleted",
            "Title": "Listing Sold",
            "Message": f"Listing {listing_id} sold via offer {accepted_offer_id}.",
            "RelatedListingID": listing_id,
            "RelatedOfferID": accepted_offer_id,
            "RelatedTransactionID": accepted_txn_id,
            "CreatedDate": iso_now(),
        }
        assert_ok(
            mgr.tx_insert(tx_id, DB_NAME, "Notification", seller_note),
            "Step 6b (notify seller)",
        )
        created_notification_ids.append(seller_note_id)

        for other_offer_id, decline_txn_id in declined_txn_pairs:
            other_offer, msg = mgr.tx_get(tx_id, DB_NAME, "Offer", other_offer_id)
            if other_offer is None:
                raise RuntimeError(msg)
            loser_note_id = next_int_key(mgr, "Notification")
            loser_note = {
                "NotificationID": loser_note_id,
                "RecipientID": other_offer["BuyerID"],
                "NotificationType": "OfferDeclined",
                "Title": "Offer Declined",
                "Message": f"Offer {other_offer_id} was declined because another offer was accepted.",
                "RelatedListingID": listing_id,
                "RelatedOfferID": other_offer_id,
                "RelatedTransactionID": decline_txn_id,
                "CreatedDate": iso_now(),
            }
            assert_ok(
                mgr.tx_insert(tx_id, DB_NAME, "Notification", loser_note),
                f"Step 6c (notify losing buyer for offer {other_offer_id})",
            )
            created_notification_ids.append(loser_note_id)

    return {
        "accepted_offer_id": accepted_offer_id,
        "listing_id": listing_id,
        "competing_offer_ids": competing_offer_ids,
        "removed_watchlist_ids": removed_watchlist_ids,
        "accepted_transaction_id": accepted_txn_id,
        "declined_transaction_pairs": declined_txn_pairs,
        "created_notification_ids": created_notification_ids,
    }

seed_members = [
    {"MemberID": 10, "Name": "Sam Seller", "Role": "Seller"},
    {"MemberID": 21, "Name": "Ben Buyer", "Role": "Buyer"},
    {"MemberID": 22, "Name": "Cara Buyer", "Role": "Buyer"},
    {"MemberID": 23, "Name": "Dina Buyer", "Role": "Buyer"},
    {"MemberID": 24, "Name": "Evan Buyer", "Role": "Buyer"},
]

seed_listings = [
    {"ListingID": 1001, "SellerID": 10, "Title": "Road Bike", "Status": "Listed", "LastModifiedDate": ""},
    {"ListingID": 1002, "SellerID": 10, "Title": "Gaming Chair", "Status": "Listed", "LastModifiedDate": ""},
]

seed_offers = [
    {"OfferID": 501, "ListingID": 1001, "BuyerID": 21, "OfferedPrice": 120.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
    {"OfferID": 502, "ListingID": 1001, "BuyerID": 22, "OfferedPrice": 117.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
    {"OfferID": 503, "ListingID": 1001, "BuyerID": 23, "OfferedPrice": 119.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
    {"OfferID": 601, "ListingID": 1002, "BuyerID": 24, "OfferedPrice": 90.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
    {"OfferID": 602, "ListingID": 1002, "BuyerID": 22, "OfferedPrice": 92.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
    {"OfferID": 603, "ListingID": 1002, "BuyerID": 23, "OfferedPrice": 95.0, "OfferStatus": "Submitted", "AgreedPrice": None, "Reason": "", "ResponseDate": ""},
]

seed_watchlist = [
    {"WatchlistID": 9001, "ListingID": 1001, "BuyerID": 22},
    {"WatchlistID": 9002, "ListingID": 1001, "BuyerID": 23},
    {"WatchlistID": 9003, "ListingID": 1001, "BuyerID": 24},
    {"WatchlistID": 9011, "ListingID": 1002, "BuyerID": 21},
    {"WatchlistID": 9012, "ListingID": 1002, "BuyerID": 22},
    {"WatchlistID": 9013, "ListingID": 1002, "BuyerID": 23},
]

print("Schemas created and helper functions loaded.")

(True, "Database 'CampusTrading' created")
(True, "Table 'Member' created in database 'CampusTrading'")
(True, "Table 'Listing' created in database 'CampusTrading'")
(True, "Table 'Offer' created in database 'CampusTrading'")
(True, "Table 'Watchlist' created in database 'CampusTrading'")
(True, "Table 'Transaction' created in database 'CampusTrading'")
(True, "Table 'Notification' created in database 'CampusTrading'")
Schemas created and helper functions loaded.


### Why This Big Setup Cell Matters

- It defines realistic schemas and seed data that mimic the Section 5.1 business flow.
- It builds one reusable function, `run_accept_offer_flow`, that executes all dependent updates in a single transaction.
- Each step has explicit success checks, so failures are surfaced immediately with context.

From this point on, every state change you see is intentional and traceable to one flow step.

## Part 3: Seed Baseline State (Committed)

Seed data is inserted inside one committed transaction so it is also present in WAL.

This is important because recovery on a fresh process replays only WAL history.

In [28]:
tx_seed = db.begin_transaction()
print(f"Seeding transaction: {tx_seed}")

for member in seed_members:
    assert_ok(db.tx_insert(tx_seed, DB_NAME, "Member", member), f"Seed Member {member['MemberID']}")

for listing in seed_listings:
    assert_ok(db.tx_insert(tx_seed, DB_NAME, "Listing", listing), f"Seed Listing {listing['ListingID']}")

for offer in seed_offers:
    assert_ok(db.tx_insert(tx_seed, DB_NAME, "Offer", offer), f"Seed Offer {offer['OfferID']}")

for watch in seed_watchlist:
    assert_ok(db.tx_insert(tx_seed, DB_NAME, "Watchlist", watch), f"Seed Watchlist {watch['WatchlistID']}")

print(db.commit_transaction(tx_seed))

print_listing_bundle(db, 1001, "Baseline BEFORE accept-offer flow (Listing 1001)")
print_listing_bundle(db, 1002, "Baseline BEFORE accept-offer flow (Listing 1002)")

Seeding transaction: T-46ad644a6e2e
Seed Member 10: Record inserted in table 'Member'
Seed Member 21: Record inserted in table 'Member'
Seed Member 22: Record inserted in table 'Member'
Seed Member 23: Record inserted in table 'Member'
Seed Member 24: Record inserted in table 'Member'
Seed Listing 1001: Record inserted in table 'Listing'
Seed Listing 1002: Record inserted in table 'Listing'
Seed Offer 501: Record inserted in table 'Offer'
Seed Offer 502: Record inserted in table 'Offer'
Seed Offer 503: Record inserted in table 'Offer'
Seed Offer 601: Record inserted in table 'Offer'
Seed Offer 602: Record inserted in table 'Offer'
Seed Offer 603: Record inserted in table 'Offer'
Seed Watchlist 9001: Record inserted in table 'Watchlist'
Seed Watchlist 9002: Record inserted in table 'Watchlist'
Seed Watchlist 9003: Record inserted in table 'Watchlist'
Seed Watchlist 9011: Record inserted in table 'Watchlist'
Seed Watchlist 9012: Record inserted in table 'Watchlist'
Seed Watchlist 9013: R

### Reading The Baseline Snapshot

- This snapshot is your control state before business logic transactions run.
- Listing 1001 and 1002 should both still be `Listed`.
- Offers should still be `Submitted` and watchlist rows should still exist.

If these baseline prints look wrong, re-run from the top before validating recovery behavior.

## Part 4: Committed Accept-Offer Flow (Listing 1001)

We now execute the full flow and COMMIT it.

Expected after commit for Listing 1001:

- Offer 501 -> Accepted
- Offers 502 and 503 -> Declined
- Listing 1001 -> Sold
- Watchlist rows for listing 1001 removed
- Transaction rows inserted (accepted + declined audit rows)
- Notifications inserted (winner, seller, losing buyers)

In [29]:
print_listing_bundle(db, 1001, "State right before COMMITTED accept-offer tx (Listing 1001)")

tx_accept_commit = db.begin_transaction()
summary_commit = run_accept_offer_flow(
    db,
    tx_id=tx_accept_commit,
    listing_id=1001,
    accepted_offer_id=501,
    acting_seller_id=10,
    include_notifications=True,
    include_declined_transactions=True,
    clear_watchlist=True,
)

print("\nFlow summary (committed path):")
pprint(summary_commit)
print(db.commit_transaction(tx_accept_commit))

print_listing_bundle(db, 1001, "State AFTER COMMITTED accept-offer tx (Listing 1001)")


State right before COMMITTED accept-offer tx (Listing 1001)

Listing snapshot for ListingID=1001
Listing: 1 row(s)
   {'ListingID': 1001, 'SellerID': 10, 'Title': 'Road Bike', 'Status': 'Listed', 'LastModifiedDate': ''}

Offer snapshot for ListingID=1001
Offer: 3 row(s)
   {'OfferID': 501, 'ListingID': 1001, 'BuyerID': 21, 'OfferedPrice': 120.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}
   {'OfferID': 502, 'ListingID': 1001, 'BuyerID': 22, 'OfferedPrice': 117.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}
   {'OfferID': 503, 'ListingID': 1001, 'BuyerID': 23, 'OfferedPrice': 119.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}

Watchlist snapshot for ListingID=1001
Watchlist: 3 row(s)
   {'WatchlistID': 9001, 'ListingID': 1001, 'BuyerID': 22}
   {'WatchlistID': 9002, 'ListingID': 1001, 'BuyerID': 23}
   {'WatchlistID': 9003, 'ListingID': 1001, 'BuyerID': 24}

Transaction

### Interpreting The Committed Flow

After commit on Listing 1001, verify all of these hold together as one atomic result:

- Accepted offer is `Accepted` with agreed price set.
- Competing submitted offers are `Declined`.
- Listing status is `Sold`.
- Watchlist rows for that listing are removed.
- Transaction and notification rows are present.

These rows should persist after restart because this transaction has a COMMIT record in WAL.

## Part 5: Crash Window Transaction (Listing 1002, Intentionally Uncommitted)

Now we run the same flow on Listing 1002 but we intentionally do NOT commit or rollback.

This simulates a crash after data changes were applied and WAL entries were written, but before transaction end.

Expected behavior after recovery: changes from this transaction should be undone.

In [30]:
print_listing_bundle(db, 1002, "State right before CRASH-WINDOW tx (Listing 1002)")

tx_crash_window = db.begin_transaction()
summary_uncommitted = run_accept_offer_flow(
    db,
    tx_id=tx_crash_window,
    listing_id=1002,
    accepted_offer_id=603,
    acting_seller_id=10,
    include_notifications=True,
    include_declined_transactions=True,
    clear_watchlist=True,
)

print("\nFlow summary (uncommitted path):")
pprint(summary_uncommitted)

state, msg = db.get_transaction_state(tx_crash_window)
print(f"Transaction state for {tx_crash_window}: {state} ({msg})")
print("Intentionally leaving this transaction ACTIVE to simulate crash before COMMIT/ROLLBACK.")

print_listing_bundle(db, 1002, "In-memory state AFTER uncommitted flow (pre-crash, Listing 1002)")


State right before CRASH-WINDOW tx (Listing 1002)

Listing snapshot for ListingID=1002
Listing: 1 row(s)
   {'ListingID': 1002, 'SellerID': 10, 'Title': 'Gaming Chair', 'Status': 'Listed', 'LastModifiedDate': ''}

Offer snapshot for ListingID=1002
Offer: 3 row(s)
   {'OfferID': 601, 'ListingID': 1002, 'BuyerID': 24, 'OfferedPrice': 90.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}
   {'OfferID': 602, 'ListingID': 1002, 'BuyerID': 22, 'OfferedPrice': 92.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}
   {'OfferID': 603, 'ListingID': 1002, 'BuyerID': 23, 'OfferedPrice': 95.0, 'OfferStatus': 'Submitted', 'AgreedPrice': None, 'Reason': '', 'ResponseDate': ''}

Watchlist snapshot for ListingID=1002
Watchlist: 3 row(s)
   {'WatchlistID': 9011, 'ListingID': 1002, 'BuyerID': 21}
   {'WatchlistID': 9012, 'ListingID': 1002, 'BuyerID': 22}
   {'WatchlistID': 9013, 'ListingID': 1002, 'BuyerID': 23}

Transaction snapshot 

### Interpreting The Crash-Window Flow

This section intentionally leaves one transaction ACTIVE. That means:

- In-memory state looks changed before crash.
- WAL has row-change entries for that transaction.
- There is no COMMIT/ROLLBACK end marker for it yet.

During recovery, these changes must be undone in reverse order (UNDO path).

## Part 6: WAL Inspection (Compact Table + Raw JSONL)

This section verifies that the log reflects the planned sequence.

First, we print a compact table (`lsn`, `tx_id`, `type`, `table`, `key`).
Then, we print the raw JSONL lines for low-level validation.

In [31]:
entries = db.wal.read_entries()
print(f"Total WAL records: {len(entries)}")

compact = [
    {
        "lsn": e.get("lsn"),
        "tx_id": e.get("tx_id"),
        "type": e.get("type"),
        "table": e.get("table"),
        "key": e.get("key"),
    }
    for e in entries
    if e.get("tx_id") is not None
    and e.get("type") in {"BEGIN", "INSERT", "UPDATE", "DELETE", "COMMIT", "ROLLBACK"}
    and e.get("lsn") is not None
]

headers = ["lsn", "tx_id", "type", "table", "key"]
if not compact:
    print("No WAL entries found.")
else:
    widths = {h: max(len(h), max(len(str(row.get(h, ""))) for row in compact)) for h in headers}

    def fmt_row(row):
        return " | ".join(str(row.get(h, "")).ljust(widths[h]) for h in headers)

    print("\nCompact WAL view")
    print(fmt_row({h: h for h in headers}))
    print("-+-".join("-" * widths[h] for h in headers))
    for row in compact:
        print(fmt_row(row))

print("\nRaw WAL JSONL lines")
raw_lines = wal_path.read_text(encoding="utf-8").splitlines()
for idx, line in enumerate(raw_lines, start=1):
    print(f"{idx:03d}: {line}")

Total WAL records: 52

Compact WAL view
lsn | tx_id          | type   | table                      | key 
----+----------------+--------+----------------------------+-----
1   | T-46ad644a6e2e | BEGIN  | None                       | None
2   | T-46ad644a6e2e | INSERT | CampusTrading.Member       | 10  
3   | T-46ad644a6e2e | INSERT | CampusTrading.Member       | 21  
4   | T-46ad644a6e2e | INSERT | CampusTrading.Member       | 22  
5   | T-46ad644a6e2e | INSERT | CampusTrading.Member       | 23  
6   | T-46ad644a6e2e | INSERT | CampusTrading.Member       | 24  
7   | T-46ad644a6e2e | INSERT | CampusTrading.Listing      | 1001
8   | T-46ad644a6e2e | INSERT | CampusTrading.Listing      | 1002
9   | T-46ad644a6e2e | INSERT | CampusTrading.Offer        | 501 
10  | T-46ad644a6e2e | INSERT | CampusTrading.Offer        | 502 
11  | T-46ad644a6e2e | INSERT | CampusTrading.Offer        | 503 
12  | T-46ad644a6e2e | INSERT | CampusTrading.Offer        | 601 
13  | T-46ad644a6e2e | INSERT | Camp

### How To Read WAL Output Here

Use the compact table first, then raw lines for deep check:

- `BEGIN` marks transaction start.
- `INSERT/UPDATE/DELETE` carry row-image payload (`before`/`after`).
- `COMMIT` means durable business result.
- Missing end marker indicates crash-window transaction.

You can correlate `tx_id` from this section directly with the recovery analysis in the next section.

## Part 7: Recovery Replay On Fresh Process

Now we simulate process restart: new DatabaseManager instance with empty memory but the same WAL file.

Steps:

1. Recreate empty schema objects.
2. Inspect recovery analysis plan.
3. Apply `recover_into` to run REDO then UNDO.
4. Print final state for both listings and validate expected outcomes.

In [32]:
db_recovered = DatabaseManager(wal_path=str(wal_path))
print(db_recovered.create_database(DB_NAME))
for table_name, table_schema in schemas.items():
    key_name = next(iter(table_schema.keys()))
    print(db_recovered.create_table(DB_NAME, table_name, table_schema, order=4, search_key=key_name))

print("\nFresh process state BEFORE recovery (should be empty tables):")
for name in schemas.keys():
    table, _ = db_recovered.get_table(DB_NAME, name)
    print(f"{name}: {len(table.get_all())} row(s)")

recovery = RecoveryManager(db_recovered.wal)
analysis = recovery.recover()
print("\nRecovery analysis plan:")
pprint(analysis)

result = recovery.recover_into(db_recovered)
print("\nRecovery apply result:")
pprint(result)

offers_1001 = [r for r in rows_for(db_recovered, "Offer") if r["ListingID"] == 1001]
offers_1002 = [r for r in rows_for(db_recovered, "Offer") if r["ListingID"] == 1002]
listing_1001 = [r for r in rows_for(db_recovered, "Listing") if r["ListingID"] == 1001][0]
listing_1002 = [r for r in rows_for(db_recovered, "Listing") if r["ListingID"] == 1002][0]

watch_1001 = [r for r in rows_for(db_recovered, "Watchlist") if r["ListingID"] == 1001]
watch_1002 = [r for r in rows_for(db_recovered, "Watchlist") if r["ListingID"] == 1002]
txn_1001 = [r for r in rows_for(db_recovered, "Transaction") if r["ListingID"] == 1001]
txn_1002 = [r for r in rows_for(db_recovered, "Transaction") if r["ListingID"] == 1002]

print("\nValidation checkpoints")
print("Listing 1001 status:", listing_1001["Status"])
print("Listing 1002 status:", listing_1002["Status"])
print("Listing 1001 offer statuses:", sorted((r["OfferID"], r["OfferStatus"]) for r in offers_1001))
print("Listing 1002 offer statuses:", sorted((r["OfferID"], r["OfferStatus"]) for r in offers_1002))
print("Watchlist count for 1001:", len(watch_1001))
print("Watchlist count for 1002:", len(watch_1002))
print("Transaction count for 1001:", len(txn_1001))
print("Transaction count for 1002:", len(txn_1002))

expected_1001 = {(501, "Accepted"), (502, "Declined"), (503, "Declined")}
expected_1002 = {(601, "Submitted"), (602, "Submitted"), (603, "Submitted")}
actual_1001 = set((r["OfferID"], r["OfferStatus"]) for r in offers_1001)
actual_1002 = set((r["OfferID"], r["OfferStatus"]) for r in offers_1002)

checks_ok = (
    listing_1001["Status"] == "Sold"
    and listing_1002["Status"] == "Listed"
    and actual_1001 == expected_1001
    and actual_1002 == expected_1002
)
print("Assertions pass?", checks_ok)
assert checks_ok, "Recovery validation failed: final state does not match expected REDO/UNDO outcome"

(True, "Database 'CampusTrading' created")
(True, "Table 'Member' created in database 'CampusTrading'")
(True, "Table 'Listing' created in database 'CampusTrading'")
(True, "Table 'Offer' created in database 'CampusTrading'")
(True, "Table 'Watchlist' created in database 'CampusTrading'")
(True, "Table 'Transaction' created in database 'CampusTrading'")
(True, "Table 'Notification' created in database 'CampusTrading'")

Fresh process state BEFORE recovery (should be empty tables):
Member: 0 row(s)
Listing: 0 row(s)
Offer: 0 row(s)
Watchlist: 0 row(s)
Transaction: 0 row(s)
Notification: 0 row(s)

Recovery analysis plan:
{'applied_redo': 0,
 'applied_undo': 0,
 'note': 'Analysis only. Use recover_into(db_manager) to apply REDO/UNDO.',
 'redo_transactions': ['T-1d15fbae3fdb', 'T-46ad644a6e2e'],
 'rolled_back_transactions': [],
 'status': 'ok',
 'total_records': 52,
 'undo_transactions': ['T-b9e20265de56']}

Recovery apply result:
{'applied_redo': 33,
 'applied_undo': 14,
 'note': 'REDO/UN

### Interpreting Recovery Results

The key evidence is the contrast between the two listings:

- Listing 1001 reflects committed business changes (REDO).
- Listing 1002 returns to pre-flow baseline (UNDO).

If assertions pass, your WAL + recovery implementation is consistent with transactional intent under crash conditions.

## Final Interpretation

What this notebook proves with observable output:

1. B+ Tree correctly organizes and serves row data for table operations.
2. Multi-table transactional flow can coordinate offer/listing/watchlist/transaction/notification changes atomically.
3. WAL captures ordered row-image operations with transaction boundaries.
4. Recovery on a fresh process deterministically applies REDO (committed) and UNDO (uncommitted).

If your output matches the final checkpoints, your engine behavior is consistent with the Module A transaction + recovery design intent.